# Solution Key — Closures and Decorators
## Decorator Lab (Parts 1–3)

Each decorator follows the pattern from the notebook: an outer function takes `func`, defines an inner `wrapper`, and returns it. Each part has a verified sample run and a grading note. A note on `functools.wraps` (good practice for real code) follows at the end.

---
## Part 1 — `printer` with a Letter-Count Decorator

The wrapper prints how many times each letter appears, then calls `printer` to print the string itself.

In [ ]:
from collections import Counter


def count_letters(func):
    def wrapper(text):
        # Count letters only (skip spaces/punctuation); case-sensitive.
        for letter, n in Counter(c for c in text if c.isalpha()).items():
            print(f'{letter!r}: {n}')
        return func(text)          # then run the original printer
    return wrapper


@count_letters
def printer(text):
    print(text)


printer('hello world')

> **Note:** the wrapper must do its work *and then* call `func(text)` (and ideally return its result). Using `Counter` is clean but a hand-rolled dict (`counts.get(ltr, 0) + 1`) is equally fine. "Letter" is loosely worded — counting every character, or filtering to `isalpha()`, are both acceptable as long as the choice is sensible.

---
## Part 2 — Ensure the Argument Is Positive

The wrapper validates the integer argument before letting the wrapped function run.

In [ ]:
def ensure_positive(func):
    def wrapper(n):
        if n <= 0:
            raise ValueError(f'expected a positive integer, got {n}')
        return func(n)
    return wrapper


@ensure_positive
def square(n):
    return n * n


print('square(5)  =', square(5))
try:
    square(-3)
except ValueError as e:
    print('square(-3) ->', e)

> **Note:** "positive" means `> 0`, so `n <= 0` is rejected (if a student treats 0 as acceptable, that's a reasonable "non-negative" reading — just confirm intent). Raising `ValueError` is idiomatic; printing a message and returning `None` is a weaker but acceptable alternative for this lab.

---
## Part 3 — Timer Decorator

Measure how long the wrapped function takes. `time.perf_counter()` is the right clock for timing (high-resolution and monotonic — unaffected by system-clock changes).

In [ ]:
import time


def timer(func):
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f'{func.__name__} took {elapsed:.6f} seconds')
        return result
    return wrapper


@timer
def sum_to(n):
    return sum(range(n))


print('result:', sum_to(1_000_000))

> **Note:** the timer must wrap `*args, **kwargs` (so it works on any function) and **return** the wrapped function's result, not just the time. The exact number printed will differ every run — that's expected. Watch for `time.time()` (lower resolution) or the long-removed `time.clock()`; `time.perf_counter()` is the modern choice.

---
## Good practice for real code: `functools.wraps`

The decorators above (and in the notebook) replace the original function with `wrapper`, so the decorated function reports the *wrapper's* identity:

In [ ]:
print('without wraps ->', square.__name__)   # 'wrapper', not 'square'

Decorating the inner `wrapper` with `functools.wraps(func)` copies the original's `__name__`, `__doc__`, and other metadata across, which keeps tracebacks, `help()`, and debugging sane:

In [ ]:
import functools


def ensure_positive(func):
    @functools.wraps(func)            # <-- the one extra line
    def wrapper(n):
        if n <= 0:
            raise ValueError(f'expected a positive integer, got {n}')
        return func(n)
    return wrapper


@ensure_positive
def cube(n):
    '''Return n cubed.'''
    return n ** 3


print('with wraps    ->', cube.__name__)   # 'cube'
print('docstring kept->', cube.__doc__)

> **Key note:** not required for the lab — the notebook intentionally keeps the first decorators bare to show the raw mechanics — but worth flagging, since every decorator the cohort writes in real frameworks (FastAPI routes, caching, retries) relies on `@functools.wraps` to preserve function identity.